[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Early Stopping &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]
X_train, y_train = train[FEATURES], train["tmin"]
X_valid, y_valid = valid[FEATURES], valid["tmin"]
X_test, y_test = test[FEATURES], test["tmin"]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' models, rebuilt here from the same made-up stations so that each answer runs on
its own.

In [2]:
xgboost_model = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, early_stopping_rounds=100,
                                 random_state=0)
xgboost_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

curve = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
curve.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)],
          eval_names=["training", "validation"])
history = curve.evals_result_

**1.** Train XGBoost with a patience of 20 in place of 100, at a learning rate of 0.05, and print its best iteration and the number of trees its `predict` uses.

In [3]:
impatient = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, early_stopping_rounds=20, random_state=0)
impatient.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
print("best iteration:", impatient.best_iteration, "| trees predict uses:", impatient.best_iteration + 1)
print("validation:", round(mean_squared_error(y_valid, impatient.predict(X_valid)), 3))

best iteration: 262 | trees predict uses: 263
validation: 2.517


The best iteration is 262 again, the same as with a patience of 100, and `predict` uses 263 trees. In this run XGBoost's validation loss never went 20 rounds without improving before its lowest point, so the shorter patience found the same round.

**2.** Train LightGBM with early stopping at a learning rate of 0.1 and at 0.02, with a patience of 100 and room for 5,000 trees, and print each model's best iteration and validation loss.

In [4]:
for rate in (0.1, 0.02):
    model = lgb.LGBMRegressor(n_estimators=5000, learning_rate=rate, random_state=0, verbose=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[lgb.early_stopping(100, verbose=False)])
    print(f"learning rate {rate}: best {model.best_iteration_} trees, "
          f"validation {model.best_score_['valid_0']['l2']:.3f}")

learning rate 0.1: best 120 trees, validation 2.546
learning rate 0.02: best 709 trees, validation 2.502


At 0.1 the best is 120 trees, with a validation loss of 2.546, and at 0.02 it is 709 trees, with 2.502, against 361 trees and 2.48 at 0.05. A smaller rate needs more trees, and from one run the three losses are too close to rank the rates.

**3.** Pass `min_delta=0.01` to `lgb.early_stopping`, so that a round counts as an improvement only if it lowers the validation loss by at least 0.01, and print the best iteration and validation loss.

In [5]:
coarse = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, random_state=0, verbose=-1)
coarse.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
           callbacks=[lgb.early_stopping(100, min_delta=0.01, verbose=False)])
print("best iteration:", coarse.best_iteration_, "| validation:", round(coarse.best_score_["valid_0"]["l2"], 3))

best iteration: 215 | validation: 2.49


With `min_delta=0.01` small improvements no longer count, so the best iteration is 215 trees, with a validation loss of 2.49, against 2.48 at 361 trees without it.

**4.** Find, in `history`, the first round at which the training loss falls below 1.0, and print the validation loss at that round beside the lowest validation loss.

In [6]:
training_curve = np.array(history["training"]["l2"])
validation_curve = np.array(history["validation"]["l2"])
first = int(np.argmax(training_curve < 1.0))
print("training loss first below 1.0 after", first + 1, "trees")
print("validation loss there:", round(validation_curve[first], 3), "| lowest:", round(validation_curve.min(), 3),
      "after", int(np.argmin(validation_curve)) + 1, "trees")

training loss first below 1.0 after 615 trees
validation loss there: 2.5 | lowest: 2.48 after 361 trees


The training loss first falls below 1.0 after 615 trees, where the validation loss is 2.5, above its lowest of 2.48 after 361 trees, so the training loss was still falling well after the validation loss had stopped.

**5.** Train XGBoost with `xgb.train`, the native interface, with `evals` and `early_stopping_rounds=100`, and show that `booster.predict` with `iteration_range=(0, booster.best_iteration + 1)` gives the same validation predictions as `xgboost_model.predict`.

In [7]:
train_matrix = xgb.DMatrix(X_train, y_train, enable_categorical=True)
valid_matrix = xgb.DMatrix(X_valid, y_valid, enable_categorical=True)
booster = xgb.train({"learning_rate": 0.05, "seed": 0}, train_matrix, num_boost_round=2000,
                    evals=[(valid_matrix, "validation")], early_stopping_rounds=100, verbose_eval=False)
predictions = booster.predict(valid_matrix, iteration_range=(0, booster.best_iteration + 1))
print("best iteration:", booster.best_iteration)
print("same as xgboost_model.predict:", np.allclose(predictions, xgboost_model.predict(X_valid)))

best iteration: 262
same as xgboost_model.predict: True


The native booster stops at the same round, 262, and its first 263 trees predict what `xgboost_model.predict` does.

---

&#8592; **Back to:** [Early Stopping](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/05-early-stopping.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
